# Explore Saved Results

Use [run_broadcast.ipynb](run_broadcast.ipynb) to plan, launch, collect and plot hardware campaigns or repeat the restored Monte Carlo convergence study. This notebook provides additional saved-data exploration. Historical data and campaign results load together; figures save only as PNG. See [README](README.md) for setup and interpretation.


In [ ]:
from collections import defaultdict
from pathlib import Path
import itertools

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from qiskit.quantum_info import state_fidelity

from broadcasting import load_run, list_runs
from broadcasting.analysis import joint_success_statistics
from scripts.generate_figures import generate_qec_crossover
from scripts.analyze_saved_hardware import load_hardware_runs
from broadcasting.plotting import plot_3d_fidelity, plot_delay_repeats, plot_hardware_scaling, plot_run_sweep, save_figure
from broadcasting.simulation import run_broadcast_no_qec, run_broadcast_qec
from broadcasting.validation import dedupe_by_job, find_duplicate_jobs, group_by_cohort

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})

ROOT = Path.cwd()
if not (ROOT / "broadcasting").is_dir():
    raise RuntimeError("Start Jupyter from the project directory.")
RESULTS_DIR = ROOT / "results"
CAMPAIGN_RESULTS_DIRS = []  # Add campaign roots outside results/ here.
FIGURE_DIR = Path("figures")
SAVE_FIGURES = False


## Load Runs


In [ ]:
all_roots = [RESULTS_DIR, *CAMPAIGN_RESULTS_DIRS]
hardware_runs = load_hardware_runs(*all_roots)
simulation_runs = [run for root in all_roots for run in list_runs(root)
                   if run["experiment_type"] == "simulation"]
runs = simulation_runs + hardware_runs
print(f"Found {len(runs)} run(s) in {RESULTS_DIR}/")
for i, run in enumerate(runs):
    sweep = run.get("sweep", {})
    qec = "QEC" if run.get("use_qec") else "no-QEC"
    opt = run.get("optimization_level")
    opt_text = f" opt={opt}" if opt is not None else ""
    print(
        f"[{i:>2}] {run['filename']}: M={run['M']} N={run['N']} {qec} "
        f"{run['experiment_type']}/{run.get('backend')}{opt_text} "
        f"{len(sweep.get('values', []))} {sweep.get('axis', '?')}-points"
    )


## Plot One Saved Run


In [ ]:
RUN_INDEX = -1

if not runs:
    print("No saved runs to plot.")
else:
    run = runs[RUN_INDEX]
    fig = plot_run_sweep(run, show=False)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / f"{Path(run['filename']).stem}.png")
    plt.show()


## Compare Saved Delay Repeats

Each job, campaign case and random-angle sample has its own panel. Receiver curves use the archived dt conversion and marginal Wilson 95% shot intervals. Unknown time conversions remain in native dt. Phase samples and repeats are never averaged together.


In [ ]:
selected = [run for run in hardware_runs if run["optimization_level"] == 3
            and len(run["sweep"]["values"]) > 1]
# Narrow selected by campaign run_id, backend or filename if desired.
if not selected:
    print("No saved opt3 delay sweeps found.")
else:
    fig = plot_delay_repeats(selected)
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "selected_delay_repeats.png")
    plt.show()


## Opt3 Hardware Fidelity Versus Number Of Receivers

The x axis is receiver count N, the y axis is fidelity, and color identifies sender count M. Each point is one job/case/theta receiver mean; vertical bars span that observation’s receiver minimum and maximum. Backend marker shapes and small horizontal offsets keep observations separate. Bars show receiver spread, not shot uncertainty. Historical device and circuit differences remain visible; no pooled size trend is inferred.


In [ ]:
if not any(run["optimization_level"] == 3 and 0 in run["sweep"]["values"]
           for run in hardware_runs):
    print("No saved opt3 zero-delay hardware runs found.")
else:
    fig = plot_hardware_scaling(hardware_runs)
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "hardware_tau0_exploration.png")
    plt.show()
    # Full identities and coordinates for every plotted point.
    display(fig.broadcasting_points)


## QEC Crossover From Pinned Saved Simulation Runs

`figures/sources.json` selects matching M,N,alpha,theta runs explicitly. Each curve uses its own saved probability grid; the inset resolves the small-p crossover. No new simulations run here.


In [ ]:
# This helper reads pinned existing records and returns the figure when no output is requested.
fig = generate_qec_crossover([])
if SAVE_FIGURES:
    save_figure(fig, FIGURE_DIR / "qec_crossover_exploration.png")
plt.show()


## Exact Vs Sampling Groups


In [ ]:
groups = defaultdict(list)
for run in runs:
    if run["experiment_type"] == "simulation" and run.get("use_qec") and run.get("sweep", {}).get("axis") == "p":
        groups[(run["M"], run["N"])].append(run)

if not groups:
    print("No QEC simulation runs found.")
else:
    ncols = min(len(groups), 3)
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows), squeeze=False, sharey=True)

    for idx, ((M, N), group) in enumerate(sorted(groups.items())):
        ax = axes[idx // ncols][idx % ncols]
        for run in group:
            p = np.asarray(run["sweep"]["values"], dtype=float)
            avg = np.asarray(run["fidelities"], dtype=float).mean(axis=1)
            if run.get("backend") == "aer_exact":
                ax.plot(p, avg, color="tab:red", linewidth=1.8, label="exact")
            elif run.get("backend") == "aer_sampling":
                ax.plot(p, avg, "--", color="tab:blue", alpha=0.45, label=f"sampling {run.get('n_samples')}")
        ax.text(0.03, 0.05, f"M={M}, N={N}", transform=ax.transAxes, fontsize=9)
        ax.axhline(0.5, color="gray", linestyle=":", alpha=0.4)
        ax.set_xlabel("Depolarizing probability p")
        if idx % ncols == 0:
            ax.set_ylabel("Average fidelity")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)

    for idx in range(len(groups), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "exact_sampling_groups.png")
    plt.show()


## Pairwise Receiver Exploration

This cell contains optional pairwise-fidelity exploration. It is off by default because the QEC contour can be slow.


In [ ]:
RUN_PAIRWISE_SCATTER = False
RUN_PAIRWISE_CONTOUR = False

if RUN_PAIRWISE_SCATTER:
    M_pair, N_pair = 2, 3
    repetitions = 2
    outcomes = list(itertools.product(range(N_pair + 1), repeat=M_pair))
    results = []
    for _ in range(repetitions):
        for outcome in outcomes:
            p_list_pair = np.random.uniform(0.0, 1.0, N_pair)
            theta_pair = np.random.uniform(0.0, 2 * np.pi, M_pair)
            results.append(
                run_broadcast_qec(
                    M=M_pair,
                    N=N_pair,
                    alpha=1 / np.sqrt(2),
                    theta_list=theta_pair,
                    p_list=p_list_pair,
                    outcomes_list=outcome,
                )
            )

    n = len(results)
    points = np.zeros((3, 3 * n))
    for i, (fidelities, _, reduced_states) in enumerate(results):
        pairs = [(0, 1), (0, 2), (1, 2)]
        for j, (a, b) in enumerate(pairs):
            points[0, i + j * n] = fidelities[a]
            points[1, i + j * n] = fidelities[b]
            points[2, i + j * n] = state_fidelity(reduced_states[a], reduced_states[b])

    fig = plot_3d_fidelity(points, show=False)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "pairwise_receiver_scatter.png")
    plt.show()

if RUN_PAIRWISE_CONTOUR:
    M_c, N_c = 1, 2
    theta_c = [np.pi / 5]
    p_grid = np.linspace(0.0, 1.0, 18)
    outcomes = list(itertools.product(range(N_c + 1), repeat=M_c))

    def collect_points(use_qec):
        runner = run_broadcast_qec if use_qec else run_broadcast_no_qec
        F1, F2, F12 = [], [], []
        for p1 in p_grid:
            for p2 in p_grid:
                f1 = f2 = f12 = 0.0
                for outcome in outcomes:
                    fidelities, _, reduced = runner(
                        M=M_c,
                        N=N_c,
                        alpha=1 / np.sqrt(2),
                        theta_list=theta_c,
                        p_list=[p1, p2],
                        outcomes_list=outcome,
                    )
                    f1 += fidelities[0]
                    f2 += fidelities[1]
                    f12 += state_fidelity(reduced[0], reduced[1])
                denom = len(outcomes)
                F1.append(f1 / denom)
                F2.append(f2 / denom)
                F12.append(f12 / denom)
        return np.asarray(F1), np.asarray(F2), np.asarray(F12)

    datasets = [collect_points(False), collect_points(True)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    for ax, (F1, F2, F12), label in zip(axes, datasets, ["No QEC", "QEC"]):
        tcf = ax.tricontourf(F1, F2, F12, levels=np.linspace(0, 1, 21), cmap=cm.magma, vmin=0, vmax=1)
        ax.text(0.03, 0.05, label, transform=ax.transAxes, color="white", fontsize=10)
        ax.set_xlabel("Fidelity(receiver 1, target)")
        ax.set_ylabel("Fidelity(receiver 2, target)")
        ax.set_xlim(0.5, 1.0)
        ax.set_ylim(0.5, 1.0)
        ax.set_aspect("equal")
        fig.colorbar(tcf, ax=ax, label="Fidelity(receiver 1, receiver 2)")
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "pairwise_receiver_contours.png")
    plt.show()


## Joint/Global Fidelity And Receiver Covariance

Each histogram preserves joint receiver outcomes. The shared helper computes global fidelity, worst-receiver fidelity, receiver-success covariance and paired receiver asymmetry with finite-shot intervals without pooling theta samples or jobs. These intervals assume independent shots with fixed probabilities and do not include device drift. For every saved job/delay, run `scripts/analyze_saved_hardware.py`; its derived report is in `analysis/hardware/report.md`.


In [ ]:
RUN_FILENAME = None  # None -> first hardware run with counts
SWEEP_INDEX = 0
THETA_INDEX = 0

candidates = [
    run for run in runs
    if run["experiment_type"] == "hardware" and run.get("counts")
    and (RUN_FILENAME is None or run["filename"] == RUN_FILENAME)
]
if not candidates:
    print("No hardware run with saved joint counts found.")
else:
    run = candidates[0]
    stats = joint_success_statistics(run["counts"][THETA_INDEX][SWEEP_INDEX], run["N"])
    print(f"Run: {run['filename']}, theta={THETA_INDEX}, delay={run['sweep']['values'][SWEEP_INDEX]} dt")
    for key, value in stats.items():
        print(f"  {key}: {value}")
    print("Nonzero success covariance does not identify correlated physical noise; "
          "preparation, feedforward, readout and within-job drift can contribute. "
          "Zero covariance in one measurement basis also does not rule out correlated noise.")
